[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB05_alignment_scoring_dotplots.ipynb)

> **Opening this notebook from Canvas:** Select **File → Save a copy in Drive** before you begin. Work in your saved copy—not in the repository preview.

**Notebook ID:** `NB05_alignment_scoring_dotplots`  
**Course release:** Fall 2026  
**Version:** 1.0  
**Updated:** 2026-09-04


# NB05 — From accepted mutations to PAM and BLOSUM

**Biological question:** How can observations from trusted protein alignments become probabilities and then alignment scores?

## Learning goals

By the end, you should be able to:

1. Explain why positional homology must be trustworthy before substitutions are counted.
2. Convert accepted-mutation counts into a transition-probability matrix.
3. Explain what repeated matrix multiplication means in a PAM model.
4. Interpret a log-odds substitution score.
5. Contrast PAM and BLOSUM data sources and numbering conventions.

**Input:** `accepted_mutations_teaching.tsv`  
**Outputs:** transition matrices, `selected_log_odds_scores.tsv`, and `pam_transition_demo.png`

> The count table is a deliberately small **synthetic teaching dataset**. It reproduces the logic of a Dayhoff-style calculation, not the historical Dayhoff PAM1 values.


## 1. Add the tools and connect Google Drive

Python is the language; Biopython, NumPy, pandas, and Matplotlib are toolkits. You do not need to memorize the code today. Read each cell's question, run it, and inspect the result.


In [ ]:
%pip install -q biopython

from pathlib import Path
from urllib.request import urlretrieve
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Bio import Align, AlignIO, Phylo, SeqIO
from Bio.Align import substitution_matrices
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

from google.colab import drive
drive.mount("/content/drive")
print("Tools and Google Drive are ready.")


## 2. Locate the course folders

The notebook recognizes the standard student folder and the instructor's `Teaching` folder. It creates only missing folders and never overwrites an existing data file.


In [ ]:
COURSE_FOLDER_NAME = "BIOINFO4-5203-F26"
COURSE_RELEASE = "Fall 2026"
REPOSITORY_BRANCH = "main"
NOTEBOOK_ID = "NB05_alignment_scoring_dotplots"
NOTEBOOK_VERSION = "1.0"

candidate_course_dirs = [
    Path("/content/drive/MyDrive") / COURSE_FOLDER_NAME,
    Path("/content/drive/MyDrive/Teaching") / COURSE_FOLDER_NAME,
]
existing_course_dirs = [path for path in candidate_course_dirs if path.exists()]
COURSE_DIR = existing_course_dirs[0] if existing_course_dirs else candidate_course_dirs[0]

DATA_DIR = COURSE_DIR / "Data" / NOTEBOOK_ID
OUTPUT_DIR = COURSE_DIR / "Outputs" / NOTEBOOK_ID
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MUTATION_PATH = DATA_DIR / "accepted_mutations_teaching.tsv"
BACKGROUND_PATH = DATA_DIR / "background_frequencies_teaching.tsv"

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/RobBurnap/"
    "Bioinformatics-MICR4203-MICR5203/"
    f"{REPOSITORY_BRANCH}/data/{NOTEBOOK_ID}/"
)

print("Course folder :", COURSE_DIR)
print("Data folder   :", DATA_DIR)
print("Output folder :", OUTPUT_DIR)


## 3. Obtain the starter data

If a starter file is missing, the public course copy is downloaded into this notebook's `Data` folder. Existing files are left untouched.


In [ ]:
if not MUTATION_PATH.exists():
    urlretrieve(DATA_BASE_URL + "accepted_mutations_teaching.tsv", MUTATION_PATH)
    print("Downloaded:", MUTATION_PATH.name)
else:
    print("Using existing:", MUTATION_PATH.name)
if not BACKGROUND_PATH.exists():
    urlretrieve(DATA_BASE_URL + "background_frequencies_teaching.tsv", BACKGROUND_PATH)
    print("Downloaded:", BACKGROUND_PATH.name)
else:
    print("Using existing:", BACKGROUND_PATH.name)

## 4. Why begin with a trusted alignment?

A substitution can be counted only if two residues are believed to occupy homologous positions. Closely related proteins, structural evidence, conserved motifs, and careful curation make that inference more defensible. A poor alignment turns alignment errors into fictitious mutations.

Dayhoff and colleagues used closely related protein families and phylogenetic reasoning to infer **accepted point mutations**—changes that occurred and persisted in a functioning protein. Modern structural alignments can also help establish positional correspondence, but they are not the sole historical source of PAM.


In [ ]:
mutations = pd.read_csv(MUTATION_PATH, sep="	", comment="#")
background = pd.read_csv(BACKGROUND_PATH, sep="	", comment="#").set_index("amino_acid")["frequency"]

assert set(mutations.columns) == {"from_aa", "to_aa", "accepted_count"}
assert np.isclose(background.sum(), 1.0)
display(mutations.head(10))
print("Accepted changes represented:", int(mutations.accepted_count.sum()))


## 5. Build a one-step transition matrix

Each row answers: “Given the starting amino acid, what is the probability of each amino acid after one small evolutionary step?” The diagonal represents no accepted change. Rows must sum to 1.

We add each listed exchange in both directions for this symmetric teaching example. Real matrix estimation requires more data, sequence weighting, phylogenetic correction, and careful normalization.


In [ ]:
AMINO_ACIDS = list("ARNDCQEGHILKMFPSTWYV")
STAY_COUNT = 1000

counts = pd.DataFrame(0.0, index=AMINO_ACIDS, columns=AMINO_ACIDS)
for aa in AMINO_ACIDS:
    counts.loc[aa, aa] = STAY_COUNT

for row in mutations.itertuples(index=False):
    counts.loc[row.from_aa, row.to_aa] += row.accepted_count
    counts.loc[row.to_aa, row.from_aa] += row.accepted_count

P1 = counts.div(counts.sum(axis=1), axis=0)
assert np.allclose(P1.sum(axis=1), 1.0)

print("One-step probabilities starting from leucine:")
display(P1.loc[["L"]].round(4))


## 6. Let evolutionary change accumulate

If `P1` describes one small step, then `P1 @ P1` describes two steps. Matrix powers extend that idea. In the historical PAM framework, PAM1 is the estimated one-step model and PAM250 is its 250-fold extrapolation—not a matrix made directly from proteins that are 250% different.


In [ ]:
powers = [1, 10, 100, 250]
transition_matrices = {
    n: pd.DataFrame(np.linalg.matrix_power(P1.to_numpy(), n), index=AMINO_ACIDS, columns=AMINO_ACIDS)
    for n in powers
}

selected_destinations = ["L", "I", "V", "F", "G"]
trajectory = pd.DataFrame({
    f"step_{n}": transition_matrices[n].loc["L", selected_destinations]
    for n in powers
})
display(trajectory.round(4))

ax = trajectory.T.plot(marker="o", figsize=(8, 4))
ax.set(xlabel="Model depth", ylabel="P(destination | starts as L)", title="A teaching Markov model spreads probability with depth")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pam_transition_demo.png", dpi=160)
plt.show()


### Interpret the probability movement

1. Why does the probability of remaining leucine decline with model depth?
2. Why do the destination probabilities gradually approach the model's long-term composition?
3. Why is extrapolation to large PAM distances useful but also uncertain?

**Your answers:**


## 7. Convert probabilities to log-odds scores

An alignment score asks whether a pairing is more likely under an evolutionary model than under a background-chance model:

`score(i,j) = log2[ P(j given i, model) / background_frequency(j) ]`

- positive: more likely than background chance
- zero: about as likely as background chance
- negative: less likely than background chance


In [ ]:
def log_odds_matrix(transition, background_frequencies):
    expected = background_frequencies.reindex(AMINO_ACIDS).to_numpy()[None, :]
    with np.errstate(divide="ignore"):
        scores = np.log2(transition.to_numpy() / expected)
    return pd.DataFrame(scores, index=AMINO_ACIDS, columns=AMINO_ACIDS)

toy_scores = {n: log_odds_matrix(transition_matrices[n], background) for n in powers}

pairs = [("I", "L"), ("D", "E"), ("K", "R"), ("W", "G"), ("C", "C")]
selected_rows = []
for aa_1, aa_2 in pairs:
    selected_rows.append({
        "pair": f"{aa_1}-{aa_2}",
        **{f"toy_step_{n}": toy_scores[n].loc[aa_1, aa_2] for n in powers},
    })

selected_log_odds = pd.DataFrame(selected_rows).replace([np.inf, -np.inf], np.nan)
display(selected_log_odds.round(2))


## 8. Compare the teaching model with published matrices

Biopython supplies real substitution matrices. Compare selected values, but remember that the scales and construction procedures differ.

| Matrix family | Starting evidence | Meaning of a larger number |
|---|---|---|
| PAM | accepted mutations inferred from closely related families, then extrapolated | **more** evolutionary distance |
| BLOSUM | substitutions observed within conserved, ungapped blocks; similar sequences are clustered | **more** stringent clustering / generally closer comparisons |

Thus PAM and BLOSUM numbers move in opposite conceptual directions. BLOSUM62 is a widely useful default, not a universal biological truth.


In [ ]:
PAM250 = substitution_matrices.load("PAM250")
BLOSUM62 = substitution_matrices.load("BLOSUM62")

published_comparison = pd.DataFrame([
    {
        "pair": f"{aa_1}-{aa_2}",
        "PAM250": PAM250[aa_1, aa_2],
        "BLOSUM62": BLOSUM62[aa_1, aa_2],
    }
    for aa_1, aa_2 in pairs
])
display(published_comparison)


## 9. Perturb the evidence

Choose one accepted mutation count, double it, and predict which transition probability and log-odds score should change most. You may edit only `PAIR_TO_DOUBLE` below, then rerun the cell.


In [ ]:
PAIR_TO_DOUBLE = ("I", "L")  # Try ("D", "E") or ("K", "R")

trial_counts = counts.copy()
a, b = PAIR_TO_DOUBLE
original = counts.loc[a, b]
trial_counts.loc[a, b] += original
trial_counts.loc[b, a] += original
trial_P1 = trial_counts.div(trial_counts.sum(axis=1), axis=0)

print(f"P({b}|{a}) before: {P1.loc[a, b]:.5f}")
print(f"P({b}|{a}) after : {trial_P1.loc[a, b]:.5f}")


## 10. Save the evidence trail

These files separate source counts, probabilities, scores, and parameters so another person can reconstruct the analysis.


In [ ]:
mutations.to_csv(OUTPUT_DIR / "accepted_mutations_used.tsv", sep="	", index=False)
P1.to_csv(OUTPUT_DIR / "toy_transition_step_1.tsv", sep="	")
transition_matrices[250].to_csv(OUTPUT_DIR / "toy_transition_step_250.tsv", sep="	")
selected_log_odds.to_csv(OUTPUT_DIR / "selected_log_odds_scores.tsv", sep="	", index=False)
published_comparison.to_csv(OUTPUT_DIR / "published_matrix_comparison.tsv", sep="	", index=False)

pd.DataFrame([
    ["notebook_id", NOTEBOOK_ID],
    ["notebook_version", NOTEBOOK_VERSION],
    ["stay_count_per_amino_acid", STAY_COUNT],
    ["teaching_data_status", "synthetic; not historical PAM1"],
], columns=["parameter", "value"]).to_csv(OUTPUT_DIR / "run_parameters.tsv", sep="	", index=False)

print("Files created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Exit ticket

In three sentences:

1. Why must an alignment be trusted before substitutions are counted?
2. What does a positive log-odds score mean?
3. Why do PAM250 and BLOSUM62 not mean “the same distance with different names”?

**Your response:**
